# Notebook 01: Quickstart & Baseline Evaluation (NLP-28)
**Đề tài:** Sửa lỗi ngữ pháp & chuẩn hóa phụ đề ASR tiếng Việt bằng Seq2Seq

Notebook này giúp bạn đi qua từng bước của toàn bộ hệ thống:
1. **Tạo nhiễu ASR (Noisy Generator):** Mô phỏng lỗi tiếng Việt phát âm / gõ phím.
2. **Baseline đối chứng (Rule & Dictionary):** Sửa lỗi theo từ điển & luật.
3. **Mô hình Seq2Seq (ViT5 / BARTpho):** Chuẩn hóa câu theo ngữ cảnh toàn câu.
4. **Đánh giá khoa học (Evaluation):** Đo lường WER, BLEU, Edit-F1.
5. **Xử lý tệp phụ đề (.SRT):** Bóc tách text, sửa lỗi và bảo toàn timestamps.

In [1]:
import sys
sys.path.append('..')

from src.data_processing.noise_generator import ASRNoiseGenerator
from src.baseline.rule_baseline import RuleAndDictionaryBaseline
from src.models.inference import Seq2SeqCorrector
from src.evaluation.metrics import evaluate_predictions
from src.srt_utils.srt_processor import SRTSubtitleProcessor

c:\Users\Zekas\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Thử nghiệm sinh nhiễu nhân tạo ASR

In [2]:
generator = ASRNoiseGenerator()
clean_sample = "Tôi đi làm ở ngân hàng và dành tiền tiết kiệm."
noisy_sample, errors = generator.apply_noise(clean_sample)

print(f"Câu gốc (Clean) : {clean_sample}")
print(f"Câu nhiễu (Noisy): {noisy_sample}")
print(f"Các lỗi đã áp dụng: {errors}")

Câu gốc (Clean) : Tôi đi làm ở ngân hàng và dành tiền tiết kiệm.
Câu nhiễu (Noisy): tôi đi lam ở ngân hàng và dành tiền tiết kiệm
Các lỗi đã áp dụng: ['drop_punctuation', 'drop_accents', 'lowercasing']


## 2. Thử nghiệm Baseline đối chứng

In [3]:
baseline = RuleAndDictionaryBaseline()
pred_baseline = baseline.predict(noisy_sample)

print(f"Đầu vào: {noisy_sample}")
print(f"Baseline: {pred_baseline}")

Đầu vào: tôi đi lam ở ngân hàng và dành tiền tiết kiệm
Baseline: Tôi đi lam ở ngân hàng và dành tiền tiết kiệm.


## 3. Thử nghiệm mô hình Seq2Seq (ViT5 / BARTpho)

In [4]:
corrector = Seq2SeqCorrector(model_name_or_path="bmd1905/vietnamese-correction-v2")
pred_seq2seq = corrector.predict(noisy_sample)

print(f"Đầu vào : {noisy_sample}")
print(f"Seq2Seq: {pred_seq2seq}")

[Seq2SeqCorrector] Đang tải mô hình: bmd1905/vietnamese-correction-v2 trên cpu...
[Seq2SeqCorrector] CẢNH BÁO: Không thể nạp mô hình (
BartphoTokenizer requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
).
[Seq2SeqCorrector] Tự động chuyển sang chế độ Rule/Dictionary Baseline.
Đầu vào : tôi đi lam ở ngân hàng và dành tiền tiết kiệm
Seq2Seq: Tôi đi lam ở ngân hàng và dành tiền tiết kiệm.


## 4. Đo lường các chỉ số hiệu năng (Metrics)

In [5]:
sources = [noisy_sample]
predictions = [pred_seq2seq]
references = [clean_sample]

metrics = evaluate_predictions(sources, predictions, references)
print("Kết quả đánh giá:", metrics)

Kết quả đánh giá: {'bleu': 74.19, 'wer': 9.09, 'edit_precision': 0.0, 'edit_recall': 0.0, 'edit_f1': 0.0}


## 5. Thử nghiệm bóc tách và chuẩn hóa tệp phụ đề .SRT

In [ ]:
sample_srt = '''1\n00:00:01,000 --> 00:00:03,500\ntoi di lam o ngan hang va gianh tien tiet kiem\n\n2\n00:00:04,000 --> 00:00:07,000\nhom nay thoi tiet rat dep\n'''

processor = SRTSubtitleProcessor(corrector_model=corrector)
cleaned_srt = processor.process_srt_string(sample_srt)
print("=== KẾT QUẢ PHỤ ĐỀ SRT SAU KHI SỬA ===")
print(cleaned_srt)

=== KẾT QUẢ PHỤ ĐỀ SRT SAU KHI SỬA ===
1
00:00:01,000 --> 00:00:03,500
Toi di lam o ngan hang va dành tiền tiet kiem.

2
00:00:04,000 --> 00:00:07,000
Hom nay thoi tiet rat dep.

